# Kaggle Digit Recognizer: 畳み込みニューラルネットワーク (CNN) で 99% を目指す！

本ノートブックは、手書き数字分類（Digit Recognizer）の総仕上げとして、画像認識の本命技術である **CNN（Convolutional Neural Network / 畳み込みニューラルネットワーク）** を構築し、スコア **99% 超え** を達成するための実践ガイドです。

### このノートブックで学ぶこと
1. **なぜCNNは画像に強いのか？:** ピクセルの絶対位置ではなく、画像の「エッジ」「角」「ループ（輪っか）」といった局所パーツを捉える仕組み
2. **画像テンソルへの変換 (Reshape):** 1次元の784列を `(28, 28, 1)` の2次元画像（縦×横×色チャンネル）に戻す
3. **CNNの標準アーキテクチャ:** 畳み込み（Conv2D）＋ プーリング（MaxPool2D）＋ ドロップアウト（Dropout）
4. **ランダムフォレストとの決定的な差:** 検証正解率 96.4% から 99.0% 超えへの劇的なスコア向上を体験する
5. **改善モデルのKaggle提出用CSV (`submission_cnn.csv`) の出力**

> **所要時間目安:** GPU利用時 約3分 / CPU利用時 約10分

## 1. ライブラリの読み込み
ディープラーニングフレームワークとして、直感的で分かりやすい **TensorFlow / Keras** を使用します。

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

print(f"TensorFlow バージョン: {tf.__version__}")
print("GPU利用可能: ", tf.config.list_physical_devices('GPU'))

## 2. データの読み込みと「画像の形（28×28×1）」への復元

> **前回の復習と今回の違い:**
>- **ロジスティック回帰 / ランダムフォレスト:** 784個の数値を「1列の平坦な表データ」として扱いました。
>- **CNN:** 上下左右のピクセルの並び順（空間構造）が重要なので、**`28ピクセル × 28ピクセル × 1チャンネル（白黒）`** の立体的な画像テンソルに変形（Reshape）します。

In [ ]:
# データディレクトリの探索
candidate_paths = [
    "/kaggle/input/competitions/digit-recognizer",
    "/kaggle/input/digit-recognizer",
    "../input/digit-recognizer",
    "./001_digit-recognizer/data",
    "./001_digit-recognizer",
    "./data",
    "."
]

data_dir = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, "train.csv")):
        data_dir = p
        break

if data_dir is None:
    found = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if found:
        data_dir = os.path.dirname(found[0])

print(f"データディレクトリ: {data_dir}")
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

# 特徴量とラベルの分離
y = train_df['label'].values
X = train_df.drop('label', axis=1).values
X_test = test_df.values

# 255.0 で正規化 (0.0 〜 1.0)
X = X / 255.0
X_test = X_test / 255.0

# ★重要: 1次元の784から「28 × 28 × 1」の画像テンソルに変形
X = X.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# 公平な比較のためシード42、80%/20%で分割
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練用画像: {X_train.shape} (枚数: {len(X_train)}, 縦: 28, 横: 28, チャンネル: 1)")
print(f"検証用画像: {X_val.shape}")
print(f"本番テスト画像: {X_test.shape}")

## 3. CNNモデルの設計（アーキテクチャ）

CNNは以下の部品をブロック状に積み重ねて作ります：

1. **Conv2D（畳み込み層）:** $3 \times 3$ の小さな虫眼鏡（フィルタ）を画像の上でスライドさせ、エッジや曲がりなどの「局所パーツ」を見つけ出します。
2. **MaxPool2D（プーリング層）:** $2 \times 2$ の領域で一番強い信号だけを残して画像を半分（$14 \times 14$、さらに $7 \times 7$）に圧縮します。これにより、**文字が少し斜めになったりズレたりしても同じ文字だと認識できる「位置不変性」** が生まれます。
3. **Dropout（ドロップアウト）:** 学習中にランダムに 25〜30% の神経結合を休ませて、特定のピクセルへの「丸暗記（過学習）」を強力に防止します。
4. **Dense（全結合層）＋ Softmax:** 抽出されたパーツ情報をもとに、最終的に「0〜9のどの確率が一番高いか」を決定します。

In [ ]:
model = keras.Sequential([
    # --- 第1ブロック (低レベル特徴: 線の傾き・エッジの抽出) ---
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(2, 2)),  # 28x28 -> 14x14
    layers.Dropout(0.25),

    # --- 第2ブロック (高レベル特徴: 角・ループ・形状の統合) ---
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=(2, 2)),  # 14x14 -> 7x7
    layers.Dropout(0.25),

    # --- 分類ヘッド (判別と出力) ---
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')  # 0〜9の10クラス確率出力
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. CNNの学習（Training）
エポック数（教科書を繰り返す回数）は 12回 とします。Kaggleの無料GPU（またはCPU）で数分で完了します。

In [ ]:
epochs = 12
batch_size = 128

print(f"CNNの学習を開始します ({epochs} エポック)...")
history = model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val),
    verbose=1
)
print("学習が完了しました！")

## 5. 学習曲線の可視化（過学習のチェック）
統計学・機械学習で最も重要な **「訓練損失（Loss）と検証損失」の推移** をプロットします。
両方の線が一緒に下がっていれば、過学習せず健全に学習が進んだ証拠です。

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 損失 (Loss) の推移
ax1.plot(history.history['loss'], label='Training Loss', marker='o')
ax1.plot(history.history['val_loss'], label='Validation Loss', marker='o')
ax1.set_title('Loss Progression (Cross-Entropy)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# 正解率 (Accuracy) の推移
ax2.plot(history.history['accuracy'], label='Training Accuracy', marker='o')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
ax2.set_title('Accuracy Progression')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 6. 検証データでの評価と3モデル最終比較！
これまでに実験した3つのモデルの正解率と誤答数を並べて比較します。

In [ ]:
# CNNによる検証データの予測
y_val_probs = model.predict(X_val)
y_val_pred_cnn = np.argmax(y_val_probs, axis=1)

cnn_acc = accuracy_score(y_val, y_val_pred_cnn)
cnn_mistakes = int((1 - cnn_acc) * len(y_val))

print(f"=== CNN 検証正解率: {cnn_acc:.4f} ({cnn_acc * 100:.2f}%) ===")
print(f"=== 検証データ 8,400問中、間違えたのはわずか {cnn_mistakes} 問！ ===")

# 3モデルの比較表
models_summary = pd.DataFrame({
    'モデル名': [
        '1. ロジスティック回帰 (線形モデル)',
        '2. ランダムフォレスト (非線形決定木)',
        '3. 畳み込みニューラルネット (CNN)'
    ],
    '検証正解率': [
        '0.9132 (91.3%)',
        '0.9639 (96.4%)',
        f'{cnn_acc:.4f} ({cnn_acc*100:.2f}%)'
    ],
    '間違えた問題数 (8,400問中)': [
        '728問',
        '302問',
        f'{cnn_mistakes}問'
    ]
})
print("\n--- 3世代のモデル進化まとめ ---")
print(models_summary.to_string(index=False))

# 比較グラフ
plt.figure(figsize=(7, 4))
bars = plt.bar(['Logistic Reg', 'Random Forest', 'CNN (Deep Learning)'], [0.9132, 0.9639, cnn_acc], color=['skyblue', 'forestgreen', 'coral'])
plt.ylim(0.88, 1.0)
plt.ylabel('Validation Accuracy')
plt.title('Evolution of Model Accuracy on Digit Recognizer')
for b in bars:
    yval = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, yval + 0.002, f"{yval*100:.2f}%", ha='center', fontweight='bold')
plt.show()

ランダムフォレストで間違えていた300問のうち、**200問以上が救済され、誤答が100問以下（約70〜90問）まで激減** しているはずです！

## 7. 提出用ファイル (`submission_cnn.csv`) の作成
テストデータ（28,000枚）を予測し、自己最高スコア（99%台）を目指す提出用CSVファイルを出力します。

In [ ]:
# 本番テストデータの予測
test_probs = model.predict(X_test)
test_preds = np.argmax(test_probs, axis=1)

# 提出DataFrameの作成
submission_df = pd.DataFrame({
    'ImageId': range(1, len(test_preds) + 1),
    'Label': test_preds
})

submission_file = 'submission_cnn.csv'
submission_df.to_csv(submission_file, index=False)

print(f"{submission_file} を出力しました！")
print("先頭5行:")
print(submission_df.head())
print(f"行数チェック: {len(submission_df)} (28000ならOK)")
print(f"欠損値チェック: {submission_df.isnull().sum().sum()} (0ならOK)")

## 8. 次のステップ: Kaggleへの提出手順
1. ノートブック右サイドバーの **Output** フォルダを展開します。
2. 生成された **`submission_cnn.csv`** を確認し、右側の3点リーダーからダウンロード（または画面下の「Submit to competition」）をクリックして提出します。
3. スコアが **`0.963` から `0.990` 前後へジャンプアップ** するのをご確認ください！